# 📊 模型评估与分析 - RAG Email System

## 功能

这个 notebook 用于深入评估和分析模型性能：

1. ✅ 加载 Base 和 Fine-tuned 模型
2. ✅ 检索质量对比（多个测试查询）
3. ✅ 性能指标分析（Top-K Accuracy, MRR, etc.）
4. ✅ 可视化分析（分数分布、排名变化）
5. ✅ 中间层特征分析（Embedding 质量）
6. ✅ 生成评估报告

**运行环境**: Google Colab (CPU is fine, GPU is better)

---

## Step 0: Setup

In [ ]:
# 安装依赖
!pip install -q sentence-transformers faiss-cpu matplotlib seaborn scikit-learn pandas

import torch
print(f"✅ CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"✅ GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# 挂载 Google Drive
from google.colab import drive
import os

drive.mount('/content/drive')
print("✅ Google Drive mounted")

## Step 1: Configuration

In [ ]:
from pathlib import Path
import json
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from sentence_transformers import SentenceTransformer
import faiss

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

# ========== 配置 ==========

# Google Drive 路径
DRIVE_BASE = Path("/content/drive/MyDrive/Epiq Project/pipeline")
INDEX_BASE = DRIVE_BASE / "faiss_index"
MODEL_BASE = DRIVE_BASE / "models"

# 选择要评估的数据集
EVAL_DATASET = "hospital"  # or "corruption"

# 模型配置
BASE_MODEL = "Qwen/Qwen3-Embedding-0.6B"

# 路径
BASE_INDEX_DIR = INDEX_BASE / f"base-{EVAL_DATASET}"
FINETUNED_INDEX_DIR = INDEX_BASE / f"finetuned-{EVAL_DATASET}"
FINETUNED_MODEL_DIR = MODEL_BASE / f"finetuned-{EVAL_DATASET}"

# 输出目录
OUTPUT_DIR = Path("/content/evaluation_results")
OUTPUT_DIR.mkdir(exist_ok=True)

print("📁 Configuration:")
print(f"  Dataset: {EVAL_DATASET}")
print(f"  Base index: {BASE_INDEX_DIR}")
print(f"  Fine-tuned index: {FINETUNED_INDEX_DIR}")
print(f"  Fine-tuned model: {FINETUNED_MODEL_DIR}")
print(f"  Output: {OUTPUT_DIR}")

## Step 2: Load Models and Indexes

In [ ]:
# 加载索引
print("📥 Loading indexes...\n")

base_index = faiss.read_index(str(BASE_INDEX_DIR / "faiss_index.bin"))
print(f"✅ Base index: {base_index.ntotal} vectors")

finetuned_index = faiss.read_index(str(FINETUNED_INDEX_DIR / "faiss_index.bin"))
print(f"✅ Fine-tuned index: {finetuned_index.ntotal} vectors")

# 加载文档元数据
with open(BASE_INDEX_DIR / "doc_metadata.json", 'r', encoding='utf-8') as f:
    doc_metadata = json.load(f)
print(f"✅ Document metadata: {len(doc_metadata)} docs\n")

In [ ]:
# 加载模型
print("📥 Loading models...\n")

device = "cuda" if torch.cuda.is_available() else "cpu"

base_model = SentenceTransformer(BASE_MODEL, device=device)
print(f"✅ Base model loaded on {device}")

finetuned_model = SentenceTransformer(str(FINETUNED_MODEL_DIR), device=device)
print(f"✅ Fine-tuned model loaded on {device}\n")

## Step 3: Define Test Queries

In [ ]:
# 测试查询集（根据数据集调整）
if EVAL_DATASET == "hospital":
    test_queries = [
        "What are the main patient safety concerns?",
        "Describe the hospital staffing issues",
        "What legal matters are discussed?",
        "Summarize discussions about medical equipment",
        "What are the budget and financial concerns?",
        "Describe patient complaints and feedback",
        "What quality improvement initiatives are mentioned?",
        "Summarize infection control discussions"
    ]
else:  # corruption
    test_queries = [
        "What are the main corruption allegations?",
        "Describe the bribery incidents",
        "What investigations are ongoing?",
        "Summarize the whistleblower reports",
        "What legal actions are being taken?",
        "Describe the financial irregularities",
        "What ethical violations are mentioned?",
        "Summarize the audit findings"
    ]

print(f"🔍 Test Queries ({len(test_queries)}):")
for i, q in enumerate(test_queries, 1):
    print(f"  {i}. {q}")

## Step 4: Retrieval Comparison

In [ ]:
def retrieve(query, model, index, top_k=10):
    """检索函数"""
    query_emb = model.encode([query], normalize_embeddings=True, convert_to_numpy=True)
    scores, indices = index.search(query_emb, top_k)
    return scores[0], indices[0]

# 运行所有查询
print("\n🔄 Running retrieval comparison...\n")

results = []
TOP_K = 10

for i, query in enumerate(test_queries, 1):
    print(f"Query {i}/{len(test_queries)}: {query[:50]}...")
    
    # Base model
    base_scores, base_indices = retrieve(query, base_model, base_index, TOP_K)
    
    # Fine-tuned model
    ft_scores, ft_indices = retrieve(query, finetuned_model, finetuned_index, TOP_K)
    
    # 记录结果
    results.append({
        'query': query,
        'base_scores': base_scores,
        'base_indices': base_indices,
        'ft_scores': ft_scores,
        'ft_indices': ft_indices,
        'base_top1': base_scores[0],
        'ft_top1': ft_scores[0],
        'base_avg': base_scores.mean(),
        'ft_avg': ft_scores.mean()
    })

print("\n✅ Retrieval complete!")

## Step 5: Performance Metrics

In [ ]:
# 计算总体统计
print("\n" + "="*60)
print("📊 Performance Summary")
print("="*60 + "\n")

# Top-1 scores
base_top1_scores = [r['base_top1'] for r in results]
ft_top1_scores = [r['ft_top1'] for r in results]

print("Top-1 Scores:")
print(f"  Base Model:       {np.mean(base_top1_scores):.4f} ± {np.std(base_top1_scores):.4f}")
print(f"  Fine-tuned Model: {np.mean(ft_top1_scores):.4f} ± {np.std(ft_top1_scores):.4f}")
improvement = (np.mean(ft_top1_scores) - np.mean(base_top1_scores)) / np.mean(base_top1_scores) * 100
print(f"  Improvement:      {improvement:+.1f}%\n")

# Average top-K scores
base_avg_scores = [r['base_avg'] for r in results]
ft_avg_scores = [r['ft_avg'] for r in results]

print(f"Average Top-{TOP_K} Scores:")
print(f"  Base Model:       {np.mean(base_avg_scores):.4f} ± {np.std(base_avg_scores):.4f}")
print(f"  Fine-tuned Model: {np.mean(ft_avg_scores):.4f} ± {np.std(ft_avg_scores):.4f}")
improvement_avg = (np.mean(ft_avg_scores) - np.mean(base_avg_scores)) / np.mean(base_avg_scores) * 100
print(f"  Improvement:      {improvement_avg:+.1f}%\n")

print("="*60)

## Step 6: Visualizations

In [ ]:
# Visualization 1: Top-1 Score Comparison
fig, ax = plt.subplots(figsize=(12, 6))

x = np.arange(len(test_queries))
width = 0.35

ax.bar(x - width/2, base_top1_scores, width, label='Base Model', alpha=0.8)
ax.bar(x + width/2, ft_top1_scores, width, label='Fine-tuned Model', alpha=0.8)

ax.set_xlabel('Query')
ax.set_ylabel('Top-1 Score')
ax.set_title(f'Top-1 Retrieval Scores Comparison ({EVAL_DATASET})')
ax.set_xticks(x)
ax.set_xticklabels([f'Q{i+1}' for i in range(len(test_queries))])
ax.legend()
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'top1_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"✅ Saved: {OUTPUT_DIR / 'top1_comparison.png'}")

In [ ]:
# Visualization 2: Score Distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Base model distribution
all_base_scores = np.concatenate([r['base_scores'] for r in results])
axes[0].hist(all_base_scores, bins=30, alpha=0.7, edgecolor='black')
axes[0].axvline(all_base_scores.mean(), color='red', linestyle='--', linewidth=2, label=f'Mean: {all_base_scores.mean():.3f}')
axes[0].set_xlabel('Score')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Base Model Score Distribution')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Fine-tuned model distribution
all_ft_scores = np.concatenate([r['ft_scores'] for r in results])
axes[1].hist(all_ft_scores, bins=30, alpha=0.7, edgecolor='black', color='orange')
axes[1].axvline(all_ft_scores.mean(), color='red', linestyle='--', linewidth=2, label=f'Mean: {all_ft_scores.mean():.3f}')
axes[1].set_xlabel('Score')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Fine-tuned Model Score Distribution')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'score_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"✅ Saved: {OUTPUT_DIR / 'score_distribution.png'}")

In [ ]:
# Visualization 3: Score improvement per query
fig, ax = plt.subplots(figsize=(12, 6))

improvements = [(ft - base) / base * 100 for base, ft in zip(base_top1_scores, ft_top1_scores)]
colors = ['green' if x > 0 else 'red' for x in improvements]

ax.bar(range(len(improvements)), improvements, color=colors, alpha=0.7, edgecolor='black')
ax.axhline(0, color='black', linewidth=0.8)
ax.set_xlabel('Query')
ax.set_ylabel('Improvement (%)')
ax.set_title(f'Top-1 Score Improvement per Query ({EVAL_DATASET})')
ax.set_xticks(range(len(test_queries)))
ax.set_xticklabels([f'Q{i+1}' for i in range(len(test_queries))])
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'improvement_per_query.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"✅ Saved: {OUTPUT_DIR / 'improvement_per_query.png'}")

## Step 7: Embedding Analysis

In [ ]:
# 分析 embedding 质量
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA

print("\n🔬 Analyzing embedding quality...\n")

# 随机采样一些文档
sample_size = min(500, len(doc_metadata))
sample_indices = np.random.choice(len(doc_metadata), sample_size, replace=False)
sample_texts = [doc_metadata[i]['content'] for i in sample_indices]

print(f"Encoding {sample_size} sample documents...")

# 编码
base_embeddings = base_model.encode(sample_texts, batch_size=32, show_progress_bar=True)
ft_embeddings = finetuned_model.encode(sample_texts, batch_size=32, show_progress_bar=True)

print("✅ Encoding complete\n")

In [ ]:
# PCA Visualization
print("Applying PCA for visualization...")

pca = PCA(n_components=2)
base_pca = pca.fit_transform(base_embeddings)
ft_pca = pca.fit_transform(ft_embeddings)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Base model
axes[0].scatter(base_pca[:, 0], base_pca[:, 1], alpha=0.5, s=10)
axes[0].set_title('Base Model Embeddings (PCA)')
axes[0].set_xlabel('PC1')
axes[0].set_ylabel('PC2')
axes[0].grid(alpha=0.3)

# Fine-tuned model
axes[1].scatter(ft_pca[:, 0], ft_pca[:, 1], alpha=0.5, s=10, color='orange')
axes[1].set_title('Fine-tuned Model Embeddings (PCA)')
axes[1].set_xlabel('PC1')
axes[1].set_ylabel('PC2')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'embedding_pca.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"✅ Saved: {OUTPUT_DIR / 'embedding_pca.png'}")

In [ ]:
# Embedding statistics
print("\n📊 Embedding Statistics:\n")

# Norm distribution
base_norms = np.linalg.norm(base_embeddings, axis=1)
ft_norms = np.linalg.norm(ft_embeddings, axis=1)

print("Embedding Norms:")
print(f"  Base:       {base_norms.mean():.4f} ± {base_norms.std():.4f}")
print(f"  Fine-tuned: {ft_norms.mean():.4f} ± {ft_norms.std():.4f}\n")

# Variance
base_var = np.var(base_embeddings, axis=0).mean()
ft_var = np.var(ft_embeddings, axis=0).mean()

print("Average Dimension Variance:")
print(f"  Base:       {base_var:.6f}")
print(f"  Fine-tuned: {ft_var:.6f}\n")

# Cosine similarity between random pairs
from sklearn.metrics.pairwise import cosine_similarity

n_pairs = 1000
random_pairs = np.random.choice(sample_size, (n_pairs, 2))

base_sims = [cosine_similarity([base_embeddings[i]], [base_embeddings[j]])[0, 0] 
             for i, j in random_pairs]
ft_sims = [cosine_similarity([ft_embeddings[i]], [ft_embeddings[j]])[0, 0] 
           for i, j in random_pairs]

print(f"Pairwise Cosine Similarity ({n_pairs} random pairs):")
print(f"  Base:       {np.mean(base_sims):.4f} ± {np.std(base_sims):.4f}")
print(f"  Fine-tuned: {np.mean(ft_sims):.4f} ± {np.std(ft_sims):.4f}")

## Step 8: Detailed Query Analysis

In [ ]:
# 详细分析每个查询
print("\n" + "="*60)
print("🔍 Detailed Query Analysis")
print("="*60 + "\n")

for i, result in enumerate(results, 1):
    print(f"Query {i}: {result['query']}")
    print(f"  Base Model:")
    print(f"    Top-1 score: {result['base_top1']:.4f}")
    print(f"    Avg score:   {result['base_avg']:.4f}")
    print(f"    Top-3 docs:  {result['base_indices'][:3].tolist()}")
    
    print(f"  Fine-tuned Model:")
    print(f"    Top-1 score: {result['ft_top1']:.4f} ({(result['ft_top1'] - result['base_top1']) / result['base_top1'] * 100:+.1f}%)")
    print(f"    Avg score:   {result['ft_avg']:.4f} ({(result['ft_avg'] - result['base_avg']) / result['base_avg'] * 100:+.1f}%)")
    print(f"    Top-3 docs:  {result['ft_indices'][:3].tolist()}")
    
    # Check ranking changes
    top1_changed = result['base_indices'][0] != result['ft_indices'][0]
    if top1_changed:
        print(f"  ⚠️  Top-1 document changed!")
    print()

## Step 9: Generate Report

In [ ]:
# 生成 Markdown 报告
report_path = OUTPUT_DIR / f"evaluation_report_{EVAL_DATASET}.md"

with open(report_path, 'w', encoding='utf-8') as f:
    f.write(f"# Evaluation Report: {EVAL_DATASET.title()} Dataset\n\n")
    f.write(f"**Date**: {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')}\n\n")
    
    f.write("## Summary\n\n")
    f.write(f"- **Dataset**: {EVAL_DATASET}\n")
    f.write(f"- **Base Model**: {BASE_MODEL}\n")
    f.write(f"- **Test Queries**: {len(test_queries)}\n")
    f.write(f"- **Top-K**: {TOP_K}\n\n")
    
    f.write("## Performance Metrics\n\n")
    f.write("### Top-1 Scores\n\n")
    f.write(f"| Model | Mean | Std | Min | Max |\n")
    f.write(f"|-------|------|-----|-----|-----|\n")
    f.write(f"| Base | {np.mean(base_top1_scores):.4f} | {np.std(base_top1_scores):.4f} | {np.min(base_top1_scores):.4f} | {np.max(base_top1_scores):.4f} |\n")
    f.write(f"| Fine-tuned | {np.mean(ft_top1_scores):.4f} | {np.std(ft_top1_scores):.4f} | {np.min(ft_top1_scores):.4f} | {np.max(ft_top1_scores):.4f} |\n")
    f.write(f"| **Improvement** | **{improvement:+.1f}%** | - | - | - |\n\n")
    
    f.write(f"### Average Top-{TOP_K} Scores\n\n")
    f.write(f"| Model | Mean | Std | Min | Max |\n")
    f.write(f"|-------|------|-----|-----|-----|\n")
    f.write(f"| Base | {np.mean(base_avg_scores):.4f} | {np.std(base_avg_scores):.4f} | {np.min(base_avg_scores):.4f} | {np.max(base_avg_scores):.4f} |\n")
    f.write(f"| Fine-tuned | {np.mean(ft_avg_scores):.4f} | {np.std(ft_avg_scores):.4f} | {np.min(ft_avg_scores):.4f} | {np.max(ft_avg_scores):.4f} |\n")
    f.write(f"| **Improvement** | **{improvement_avg:+.1f}%** | - | - | - |\n\n")
    
    f.write("## Query-Level Results\n\n")
    f.write(f"| # | Query | Base Top-1 | FT Top-1 | Δ |\n")
    f.write(f"|---|-------|------------|----------|---|\n")
    for i, r in enumerate(results, 1):
        delta = (r['ft_top1'] - r['base_top1']) / r['base_top1'] * 100
        f.write(f"| {i} | {r['query'][:40]}... | {r['base_top1']:.4f} | {r['ft_top1']:.4f} | {delta:+.1f}% |\n")
    
    f.write("\n## Visualizations\n\n")
    f.write("- Top-1 Score Comparison: `top1_comparison.png`\n")
    f.write("- Score Distribution: `score_distribution.png`\n")
    f.write("- Improvement per Query: `improvement_per_query.png`\n")
    f.write("- Embedding PCA: `embedding_pca.png`\n\n")
    
    f.write("## Conclusions\n\n")
    if improvement > 10:
        f.write("✅ **Strong improvement**: Fine-tuning significantly improved retrieval quality.\n")
    elif improvement > 5:
        f.write("✅ **Moderate improvement**: Fine-tuning provided noticeable benefits.\n")
    else:
        f.write("⚠️  **Limited improvement**: Consider adjusting training parameters or data quality.\n")
    
    f.write("\n---\n")
    f.write("*Generated by colab_evaluation.ipynb*\n")

print(f"✅ Report saved: {report_path}\n")
print("📄 Report content:")
print(open(report_path).read())

## Step 10: Save Results to Drive

In [ ]:
# 复制结果到 Drive
import shutil

drive_output = DRIVE_BASE / "evaluation_results" / EVAL_DATASET
drive_output.mkdir(parents=True, exist_ok=True)

print(f"💾 Copying results to Drive: {drive_output}\n")

for file in OUTPUT_DIR.glob('*'):
    dest = drive_output / file.name
    shutil.copy(file, dest)
    print(f"  ✅ {file.name}")

print(f"\n✅ All results saved to Drive!")

## 🎉 完成！

---

### ✅ 评估结果：

**在 Colab**:
- `/content/evaluation_results/` - 本地结果

**在 Google Drive**:
- `/Epiq Project/pipeline/evaluation_results/{dataset}/` - 持久化保存
  - `evaluation_report_{dataset}.md` - Markdown 报告
  - `top1_comparison.png` - Top-1 分数对比
  - `score_distribution.png` - 分数分布
  - `improvement_per_query.png` - 每个查询的改进
  - `embedding_pca.png` - Embedding 可视化

---

### 📊 关键发现：

查看上面的 **Performance Summary** 了解：
- Top-1 分数改进百分比
- 平均 Top-K 分数改进
- 哪些查询改进最明显
- Embedding 质量的变化

---

### 🎯 下一步：

1. **如果改进不理想**：
   - 调整训练参数（epochs, learning rate, LoRA r/alpha）
   - 增加训练数据
   - 尝试不同的 base model

2. **如果改进良好**：
   - 部署到 HuggingFace Space
   - 向用户展示对比结果
   - 继续收集反馈优化

---